# GSDensity base scores: MERFISH and imaging mass cytometry

Per-cell GSDensity relevance (MCA co-embedding, 300-NN graph, random walk with restart; Liang et al. 2023) for the
MERFISH hypothalamus (`04_merfish_platform`, Fig. 5g-j) and the IMC breast cancer image (`10_masscytometry_platform_deg`,
Fig. 5k-n, and the earlier protein-marker version `10_masscytometry_platform`), with the held-out signatures of
`R4_F2f_downstream_platforms`. GSDensity runs in the R environment `gsdensity_env` (`scripts/run_gsdensity.R`) with
one MCA embedding per MERFISH section (and one on all sections pooled) and per IMC input. The relevance tables are
written to `data/gsd_cache/merfish/` and `data/gsd_cache/imc/`, where the figure notebooks read them.

In [1]:
import os
import warnings

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

In [2]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import re

from sklearn.metrics import roc_auc_score, average_precision_score

In [3]:
sc.set_figure_params(frameon=False, fontsize=8)
sc.settings._vector_friendly = False
plt.rcParams["axes.grid"] = False

In [4]:
# set working directory
# repository root (the folder containing `notebooks/`); safe to re-run this cell
import sys
project_dir = os.getcwd()
if os.path.basename(project_dir) == "notebooks":
    project_dir = os.path.dirname(project_dir)
project_dir += "/"
sys.path.insert(0, project_dir + "notebooks")  # helpers.py and score_signatures.py
os.makedirs(project_dir + "logs", exist_ok=True)
working_dir = project_dir
os.chdir(working_dir)

# set figure directory
figure_dir = working_dir + "figures/"
os.makedirs(figure_dir, exist_ok=True)

# processed data directory
processed_data = project_dir + "processed_data/"

# source data directory
source_dir = working_dir + "source_data/"



In [5]:
spot_size = 20

In [6]:
from helpers import (
    export_gsdensity_task_gsdbase,
    nonconstant_features,
    pseudobulk_deseq2_signature,
    read_gsdensity_outputs_gsdbase,
    start_gsdensity_session,
)

In [7]:
# 1. Load data as in the original notebooks
# merfish (04_merfish_platform): integer counts
adata_mf = sq.datasets.merfish(path=project_dir + "data/anndata/merfish.h5ad")
adata_mf.X = adata_mf.X.astype(int)
section_mf = adata_mf.obs["batch"].astype(str).values
sections = sorted(np.unique(section_mf), key=int)

# imc, Fig. 5k-n (10_masscytometry_platform_deg): raw intensities rounded to integers
adata_imc = sq.datasets.imc(path=project_dir + "data/anndata/imc.h5ad")
adata_imc.X = adata_imc.raw.X.round().astype(np.int64).copy()

# imc, earlier protein-marker version (10_masscytometry_platform): raw intensities as stored
adata_imc_protein = sq.datasets.imc(path=project_dir + "data/anndata/imc.h5ad")
adata_imc_protein.X = adata_imc_protein.raw.X.copy()

In [8]:
# 2. Pseudo-bulk DESeq2 signatures (03_xenium_platform, 04_merfish_platform, 10_masscytometry_platform_deg)
# signatures of the original notebooks and the held-out versions of R4_F2f_downstream_platforms
# (held out: DEGs from the even-numbered MERFISH sections, and from the IMC cells left of the x-median)
is_ep = (adata_mf.obs["Cell_class"] == "Ependymal").values
even = adata_mf.obs["batch"].astype(int).values % 2 == 0
is_basal = (adata_imc.obs["cell type"] == "basal CK tumor cell").values
is_t = (adata_imc.obs["cell type"] == "T cells").values
left = adata_imc.obsm["spatial"][:, 0] < np.median(adata_imc.obsm["spatial"][:, 0])

signatures_dict = {
    "merfish": {
        "Ependymal": pseudobulk_deseq2_signature(adata_mf, is_ep, "Ependymal", 0.01, 2.5, 20),
        "Ependymal_heldout_even": pseudobulk_deseq2_signature(adata_mf[even], is_ep[even], "Ependymal", 0.01, 2.5, 20),
    },
    "imc_deg": {
        "Basal tumour": pseudobulk_deseq2_signature(adata_imc, is_basal, "Basal tumour", 0.05, 0.5, 5),
        "T cell": pseudobulk_deseq2_signature(adata_imc, is_t, "T cells", 0.05, 0.5, 5),
        "Basal tumour_heldout_left": pseudobulk_deseq2_signature(adata_imc[left], is_basal[left], "Basal tumour", 0.05, 0.5, 5),
        "T cell_heldout_left": pseudobulk_deseq2_signature(adata_imc[left], is_t[left], "T cells", 0.05, 0.5, 5),
    },
}

# log-normalisation, as before scoring in the original notebooks
for adata in [adata_mf, adata_imc, adata_imc_protein]:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

# protein-marker version (10_masscytometry_platform): top-10 Wilcoxon markers; basal CK tumour cells by the same rule
sc.tl.rank_genes_groups(adata_imc_protein, groupby="cell type", method="wilcoxon")
sc.tl.filter_rank_genes_groups(adata_imc_protein, min_in_group_fraction=0.5, min_fold_change=2, max_out_group_fraction=0.5)
names = adata_imc_protein.uns["rank_genes_groups"]["names"]
signatures_dict["imc_protein"] = {
    "apoptotic_tumour": names["apoptotic tumor cell"][:10].tolist(),
    "t_cell": names["T cells"][:10].tolist(),
    "basal_tumour": names["basal CK tumor cell"][:10].tolist(),
}
for inp, sigs in signatures_dict.items():
    for k, g in sigs.items():
        print(f"{inp} | {k} ({len(g)}): {', '.join(g)}")

Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.00 seconds.



Fitting dispersions...


... done in 0.25 seconds.

Fitting dispersion trend curve...
... done in 0.02 seconds.

Fitting MAP dispersions...


... done in 0.19 seconds.

Fitting LFCs...
... done in 0.11 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.

Running Wald tests...


... done in 0.09 seconds.



... (15 outputs left out)

... done in 0.07 seconds.

Fitting dispersion trend curve...
... done in 0.01 seconds.

Fitting MAP dispersions...
... done in 0.10 seconds.

Fitting LFCs...
... done in 0.06 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


... done in 0.07 seconds.



Log2 fold change & Wald test p-value: group T cells vs Rest
                            baseMean  log2FoldChange     lfcSE       stat  \
1021522Tm169Di EGFR       135.369924       -0.277942  0.210063  -1.323138   
1031747Er167Di ECadhe     700.697857       -0.811663  0.126576  -6.412440   
112475Gd156Di Estroge       3.638098       -1.967800  1.467620  -1.340810   
117792Dy163Di GATA3         3.672995       -0.617695  1.078460  -0.572757   
1261726In113Di Histone   2118.781633       -0.287803  0.155385  -1.852197   
1441101Er168Di Ki67        24.315034       -2.564531  0.704311  -3.641190   
174864Nd148Di SMA         511.718611        1.794112  0.220516   8.135967   
1921755Sm149Di Vimenti    900.563843        2.398659  0.245796   9.758722   
198883Yb176Di cleaved     269.882416        0.242616  0.160156   1.514871   
201487Eu151Di cerbB       453.898337       -0.851543  0.151023  -5.638500   
234832Lu175Di panCyto     167.484557       -2.408814  0.414170  -5.815995   
3111576Nd143Di C

merfish | Ependymal (16): Nnat, Cd24a, Cckbr, Cyr61, Sln, Mlc1, Aqp4, Lpar1, Ccnd2, Greb1, Gem, Plin3, Gbx2, Sox6, Baiap2, Grpr
merfish | Ependymal_heldout_even (16): Nnat, Cd24a, Cckbr, Cyr61, Sln, Mlc1, Aqp4, Lpar1, Ccnd2, Gbx2, Plin3, Greb1, Sox6, Baiap2, Gem, Grpr
imc_deg | Basal tumour (5): 651779Pr141Di Cytoker, 346876Sm147Di Keratin, 98922Yb174Di Cytoker, 3111576Nd143Di Cytoker, 234832Lu175Di panCyto
imc_deg | T cell (5): 8001752Sm152Di CD3epsi, 71790Dy162Di CD45, 1921755Sm149Di Vimenti, 174864Nd148Di SMA, 77877Nd146Di CD68
imc_deg | Basal tumour_heldout_left (4): 346876Sm147Di Keratin, 3111576Nd143Di Cytoker, 234832Lu175Di panCyto, phospho S6
imc_deg | T cell_heldout_left (5): 8001752Sm152Di CD3epsi, 71790Dy162Di CD45, 1921755Sm149Di Vimenti, 174864Nd148Di SMA, 77877Nd146Di CD68
imc_protein | apoptotic_tumour (10): 1031747Er167Di ECadhe, 201487Eu151Di cerbB, 312878Gd158Di Progest, 1021522Tm169Di EGFR, 92964Er166Di Carboni, 1261726In113Di Histone, phospho mTOR, 112475Gd156Di Est

In [9]:
# 3. Distinct gene sets per input (GSDensity has no gene weights: identical gene sets give identical scores)
inputs = {"merfish": adata_mf, "imc_deg": adata_imc, "imc_protein": adata_imc_protein}
gene_set_runs, run_of = {}, {}
for inp, sigs in signatures_dict.items():
    seen = {}
    for k, g in sigs.items():
        present = [x for x in dict.fromkeys(g) if x in inputs[inp].var_names]
        key = frozenset(present)
        if key not in seen:
            seen[key] = k
            gene_set_runs.setdefault(inp, {})[k] = present
        run_of[(inp, k)] = seen[key]
print(pd.DataFrame([{"input": i, "signature": k, "gene set run": v} for (i, k), v in run_of.items()]).to_string(index=False))

      input                 signature              gene set run
    merfish                 Ependymal                 Ependymal
    merfish    Ependymal_heldout_even                 Ependymal
    imc_deg              Basal tumour              Basal tumour
    imc_deg                    T cell                    T cell
    imc_deg Basal tumour_heldout_left Basal tumour_heldout_left
    imc_deg       T cell_heldout_left                    T cell
imc_protein          apoptotic_tumour          apoptotic_tumour
imc_protein                    t_cell                    t_cell
imc_protein              basal_tumour              basal_tumour


In [10]:
# 4. GSDensity bridge: export a task directory, run scripts/run_gsdensity.R in gsdensity_env, read the results
gsd_dir = working_dir + "data/gsd_cache/"
gsd_script = working_dir + "scripts/run_gsdensity.R"
gsd_params = {"nn_use": 300, "restart": 0.75, "dims": 10, "seed": 0, "spatial": True, "spatial_n": 10,
              "spatial_ntimes": 100}

In [11]:
# 5. Export one task per MCA embedding: each MERFISH section, all MERFISH sections pooled, and each IMC input
tasks = {}  # tag -> (input, task directory, cells)
for b in sections:
    tasks[f"merfish_section_{b}"] = ("merfish", gsd_dir + f"merfish/_tasks/section_{b}/", section_mf == b)
tasks["merfish_pooled"] = ("merfish", gsd_dir + "merfish/_tasks/pooled/", np.ones(adata_mf.n_obs, bool))
tasks["imc_deg"] = ("imc_deg", gsd_dir + "imc/_tasks/deg/", np.ones(adata_imc.n_obs, bool))
tasks["imc_protein"] = ("imc_protein", gsd_dir + "imc/_tasks/protein/", np.ones(adata_imc.n_obs, bool))

for tag, (inp, td, m) in tasks.items():
    a = inputs[inp][m]
    if tag.startswith("merfish_section"):
        # physical coordinates (um) of the section for the spatial KLD; the relevance does not use coordinates
        export_gsdensity_task_gsdbase(a, gene_set_runs[inp], td, gsd_params, xy=a.obs[["Centroid_X", "Centroid_Y"]].values.astype(float))
    elif tag == "merfish_pooled":
        # the 12 rescaled sections overlap in one frame: no spatial KLD
        export_gsdensity_task_gsdbase(a, gene_set_runs[inp], td, {**gsd_params, "spatial": False})
    else:
        export_gsdensity_task_gsdbase(a, gene_set_runs[inp], td, gsd_params)
    print(f"{tag}: {a.n_obs:,} cells x {a.n_vars} features, {len(gene_set_runs[inp])} gene set(s)")

merfish_section_0: 6,509 cells x 161 features, 1 gene set(s)


merfish_section_1: 6,412 cells x 161 features, 1 gene set(s)


merfish_section_2: 6,507 cells x 161 features, 1 gene set(s)


merfish_section_3: 6,605 cells x 161 features, 1 gene set(s)


merfish_section_4: 6,185 cells x 161 features, 1 gene set(s)


merfish_section_5: 6,154 cells x 161 features, 1 gene set(s)


... (5 outputs left out)

merfish_section_11: 5,584 cells x 161 features, 1 gene set(s)


merfish_pooled: 73,655 cells x 161 features, 1 gene set(s)


imc_deg: 4,668 cells x 34 features, 3 gene set(s)


imc_protein: 4,668 cells x 34 features, 3 gene set(s)


In [12]:
# 6. Run GSDensity: pooled MERFISH in one R session, every other task in a second one
# (raise the 500 MB transfer limit of R's future package, which stops run.rwr.list on large graphs)
os.environ["R_FUTURE_GLOBALS_MAXSIZE"] = str(8 * 1024**3)
log_dir = working_dir + "logs/"
r_sessions = {
    "A": [tasks["merfish_pooled"][1]],
    "B": [td for tag, (inp, td, m) in tasks.items() if tag != "merfish_pooled"],
}
procs = {s: start_gsdensity_session(tds, log_dir + f"REV_GSD_base_merfish_imc_R{s}.log", gsd_script)
         for s, tds in r_sessions.items()}
print({s: p.wait() for s, p in procs.items()})

{'A': 0, 'B': 0}


In [13]:
# 7. Read the results: relevance sums to 1 per run; zero relevance (dnet::dRWR sets p < 1e-6 to 0) and NA labels
gsd = {}
summary_rows = []
for tag, (inp, td, m) in tasks.items():
    gsd[tag] = read_gsdensity_outputs_gsdbase(td, inputs[inp].obs_names[m].astype(str))
    for k in gene_set_runs[inp]:
        summary_rows.append({
            "task": tag,
            "gene set": k,
            "n cells": int(m.sum()),
            "sum relevance": gsd[tag]["scores"][k].sum(),
            "n zero relevance": int((gsd[tag]["scores"][k] == 0).sum()),
            "n positive (antimode)": int((gsd[tag]["labels"][k] == "positive").sum()),
            "n label NA": int(gsd[tag]["labels"][k].isna().sum()),
            "runtime (s)": round(gsd[tag]["timing"]["total"], 1),
        })
print(pd.DataFrame(summary_rows).round(4).to_string(index=False))

              task                  gene set  n cells  sum relevance  n zero relevance  n positive (antimode)  n label NA  runtime (s)
 merfish_section_0                 Ependymal     6509            1.0               115                     50           0         14.5
 merfish_section_1                 Ependymal     6412            1.0               156                    154           0         13.3
 merfish_section_2                 Ependymal     6507            1.0                 4                     85           0         15.1
 merfish_section_3                 Ependymal     6605            1.0               111                      0        6605         13.7
 merfish_section_4                 Ependymal     6185            1.0                25                    168           0         11.6
 merfish_section_5                 Ependymal     6154            1.0                74                      0        6154          9.8
 merfish_section_6                 Ependymal     6111  

In [14]:
# 8. Genes used per run: signature genes that vary across the cells of the run (RunMCA drops constant features)
varying = {tag: nonconstant_features(inputs[inp][m]) for tag, (inp, td, m) in tasks.items()}

In [15]:
# 9. One relevance table per signature (spot_id, slide, relevance, label, relevance_scaled) and one index per dataset
# relevance_scaled = relevance x cells in the run (mean 1 per run), so that MERFISH sections can be pooled
outputs = {  # dataset -> [(file name, input, tasks, mode)]
    "merfish": [
        ("Ependymal", "merfish", [f"merfish_section_{b}" for b in sections], "per section"),
        ("Ependymal_heldout_even", "merfish", [f"merfish_section_{b}" for b in sections], "per section"),
        ("Ependymal_pooled", "merfish", ["merfish_pooled"], "pooled"),
        ("Ependymal_heldout_even_pooled", "merfish", ["merfish_pooled"], "pooled"),
    ],
    "imc": [(k, "imc_deg", ["imc_deg"], "deg") for k in signatures_dict["imc_deg"]]
    + [(k, "imc_protein", ["imc_protein"], "protein") for k in signatures_dict["imc_protein"]],
}
tables = {}
for dataset, outs in outputs.items():
    index_rows = []
    for name, inp, tags, mode in outs:
        sig = name.replace("_pooled", "")
        k = run_of[(inp, sig)]
        parts = []
        for tag in tags:
            s = gsd[tag]["scores"][k]
            parts.append(pd.DataFrame({
                "spot_id": s.index,
                "slide": tag.replace("merfish_section_", "") if tag.startswith("merfish_section") else "all",
                "relevance": s.values,
                "label": gsd[tag]["labels"][k].values,
                "relevance_scaled": s.values * len(s),
            }))
        tab = pd.concat(parts).set_index("spot_id").reindex(inputs[inp].obs_names.astype(str))
        tab.index.name = "spot_id"
        tab = tab.reset_index()
        tab.to_csv(gsd_dir + f"{dataset}/{name}.csv", index=False)
        tables[(dataset, name)] = tab
        given = list(dict.fromkeys(signatures_dict[inp][sig]))
        used = [g for g in given if all(g in varying[t] for t in tags)]
        index_rows.append({
            "dataset": dataset,
            "signature": name,
            "gene_set_run": k,
            "direction": "unsigned",
            "genes_used": len(used),
            "genes_used_of_given": f"{len(used)}/{len(given)}",
            "n_spots": len(tab),
            "n_positive_label": int((tab["label"] == "positive").sum()),
            "n_label_NA": int(tab["label"].isna().sum()),
            "n_zero_relevance": int((tab["relevance"] == 0).sum()),
            "runtime_s": round(sum(gsd[t]["timing"]["total"] for t in tags), 1),
            "params": json.dumps({**gsd_params, "spatial": mode != "pooled"}),
            "file": f"data/gsd_cache/{dataset}/{name}.csv",
            "mode": mode,
            "mca_runs": len(tags),
            "genes_used_list": ";".join(used),
            "genes_not_used": ";".join(g for g in given if g not in used),
        })
    index = pd.DataFrame(index_rows)
    index.to_csv(gsd_dir + f"INDEX_{dataset}.csv", index=False)
    print(index[["signature", "gene_set_run", "genes_used_of_given", "n_spots", "n_positive_label", "n_label_NA",
                 "n_zero_relevance", "runtime_s"]].to_string(index=False))

                    signature gene_set_run genes_used_of_given  n_spots  n_positive_label  n_label_NA  n_zero_relevance  runtime_s
                    Ependymal    Ependymal               15/16    73655               678       24625               863      131.9
       Ependymal_heldout_even    Ependymal               15/16    73655               678       24625               863      131.9
             Ependymal_pooled    Ependymal               16/16    73655                 0       73655             60963      229.1
Ependymal_heldout_even_pooled    Ependymal               16/16    73655                 0       73655             60963      229.1


                signature              gene_set_run genes_used_of_given  n_spots  n_positive_label  n_label_NA  n_zero_relevance  runtime_s
             Basal tumour              Basal tumour                 5/5     4668               302           0              1117       10.5
                   T cell                    T cell                 5/5     4668               569           0               839       10.5
Basal tumour_heldout_left Basal tumour_heldout_left                 4/4     4668              1227           0               468       10.5
      T cell_heldout_left                    T cell                 5/5     4668               569           0               839       10.5
         apoptotic_tumour          apoptotic_tumour               10/10     4668              1542           0                 0       11.2
                   t_cell                    t_cell               10/10     4668              1254           0                 0       11.2
             basal_t

In [16]:
# 10. Pathway-level spatial KLD (compute.spatial.kld, 10 x 10 grid, 100 cell shuffles); not run for pooled MERFISH
kld_rows = []
for tag, (inp, td, m) in tasks.items():
    if gsd[tag]["spatial_kld"] is None:
        continue
    for k in gene_set_runs[inp]:
        kld_rows.append({"task": tag, "gene_set": k, **gsd[tag]["spatial_kld"].loc[k].to_dict()})
kld = pd.DataFrame(kld_rows)
print(kld.round(4).to_string(index=False))

              task                  gene_set  spatial.kld  rspatial.kld  rspatial.kld.sd  p.value  delta.kld  p.adj
 merfish_section_0                 Ependymal     -12.9265      -15.2010           0.1801      0.0     2.2745    0.0
 merfish_section_1                 Ependymal     -12.8293      -15.2682           0.1816      0.0     2.4389    0.0
 merfish_section_2                 Ependymal     -12.7997      -15.5101           0.1788      0.0     2.7104    0.0
 merfish_section_3                 Ependymal     -12.7974      -15.4294           0.1715      0.0     2.6320    0.0
 merfish_section_4                 Ependymal     -12.9593      -15.5334           0.1867      0.0     2.5741    0.0
 merfish_section_5                 Ependymal     -12.9336      -15.4242           0.1825      0.0     2.4906    0.0
 merfish_section_6                 Ependymal     -12.4081      -15.3434           0.1843      0.0     2.9353    0.0
 merfish_section_7                 Ependymal     -12.5627      -15.4367 

In [17]:
# 11. AUROC and AUPRC of the relevance against the cell labels (all cells, and the held-out half)
targets = {
    "Ependymal": ("Cell_class", "Ependymal"),
    "Basal tumour": ("cell type", "basal CK tumor cell"),
    "T cell": ("cell type", "T cells"),
    "apoptotic_tumour": ("cell type", "apoptotic tumor cell"),
    "t_cell": ("cell type", "T cells"),
    "basal_tumour": ("cell type", "basal CK tumor cell"),
}
summary_rows = []
for (dataset, name), tab in tables.items():
    base = re.sub(r"_(heldout_even|heldout_left)$", "", name.replace("_pooled", ""))
    col, pos = targets[base]
    a = adata_mf if dataset == "merfish" else adata_imc
    y = a.obs[col].astype(str).eq(pos).values
    score = tab["relevance_scaled"].values
    evals = {"all cells": np.ones(len(y), bool)}
    if dataset == "merfish":
        evals["odd sections (held-out design)"] = ~even
    else:
        evals["right of x-median (held-out design)"] = ~left
    for ev, m in evals.items():
        summary_rows.append({
            "dataset": dataset,
            "signature": name,
            "cells": ev,
            "n target": int(y[m].sum()),
            "AUROC": roc_auc_score(y[m], score[m]),
            "AUPRC": average_precision_score(y[m], score[m]),
            "prevalence": y[m].mean(),
            "% antimode positive": 100 * (tab["label"].values[m] == "positive").mean(),
        })
print(pd.DataFrame(summary_rows).round(3).to_string(index=False))

dataset                     signature                               cells  n target  AUROC  AUPRC  prevalence  % antimode positive
merfish                     Ependymal                           all cells      1961  0.961  0.512       0.027                0.921
merfish                     Ependymal      odd sections (held-out design)       931  0.961  0.502       0.025                0.709
merfish        Ependymal_heldout_even                           all cells      1961  0.961  0.512       0.027                0.921
merfish        Ependymal_heldout_even      odd sections (held-out design)       931  0.961  0.502       0.025                0.709
merfish              Ependymal_pooled                           all cells      1961  0.905  0.190       0.027                0.000
merfish              Ependymal_pooled      odd sections (held-out design)       931  0.899  0.172       0.025                0.000
merfish Ependymal_heldout_even_pooled                           all cells      1961